# Extract frames from videos

Samples frames from every video in `/corpus/raw` using one or more **extraction strategies**, so that strategies can be compared in later analyses.

| Method | Picks | Good for |
|---|---|---|
| `every_n` | Every n-th frame | Simple baseline, but depends on the frame rate |
| `every_seconds` | One frame every n seconds | Comparable across videos |
| `scene_change` | The first frame after a sudden visual change, relative to the motion around it | Edited videos with cuts |
| `uniform` | A fixed number of evenly spaced frames | VLMs with a fixed image budget |
| `keyframes` | The video's own encoded keyframes | Very fast, but depends on the encoder |
| `sharpest` | The sharpest frame in each time window | Avoids motion blur, useful before OCR |
| `cover` | The TikTok cover image | What the platform shows as the preview |

Any strategy can add `"dedupe": 5`, which drops frames nearly identical to the last one kept.

Runs on the CPU, so no GPU workspace is needed. Safe to re-run: finished videos are skipped per strategy.

In [ ]:
%pip install -q --no-warn-script-location opencv-python-headless

# Makes a package installed just now importable without restarting the kernel.
import importlib, site
site.addsitedir(site.getusersitepackages())
importlib.invalidate_caches()

In [ ]:
import json, os, shutil, subprocess, time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

RAW = Path(os.environ.get("SEMINAR_RAW", "/corpus/raw"))
OUTPUT = Path(os.environ.get("SEMINAR_OUTPUT", "/corpus/derived"))
FRAMES = OUTPUT / "frames"

# Strategy name -> settings. The name becomes the output folder, so give each
# variant its own name, for example every_5s next to every_2s.
CONFIGS = {
    "every_30f": {"method": "every_n", "n": 30},
    "every_2s": {"method": "every_seconds", "seconds": 2},
    "every_2s_dedupe": {"method": "every_seconds", "seconds": 2, "dedupe": 5},
    "scene": {"method": "scene_change", "threshold": 18, "ratio": 3, "min_gap": 0.5},
    "uniform_8": {"method": "uniform", "count": 8},
    "keyframes": {"method": "keyframes"},
    "sharpest_2s": {"method": "sharpest", "seconds": 2},
    "cover": {"method": "cover"},
}
# Which strategies to run in this session.
RUN = list(CONFIGS)

JPEG_QUALITY = 90
# Safety limit. Longer selections are thinned out evenly.
MAX_FRAMES_PER_VIDEO = 300
# Frames per second analysed by scene_change and sharpest.
ANALYSIS_FPS = 6
# Limit for a trial run, for example 3. None processes everything.
MAX_VIDEOS = int(os.environ["SEMINAR_MAX_VIDEOS"]) if os.environ.get("SEMINAR_MAX_VIDEOS") else None
VIDEO_SUFFIXES = {".mp4", ".mov", ".webm", ".mkv", ".m4v"}

## Find the videos

In [ ]:
videos = sorted(
    path for path in RAW.rglob("*")
    if path.is_file() and path.suffix.lower() in VIDEO_SUFFIXES and not path.name.startswith(".")
)
if MAX_VIDEOS:
    videos = videos[:MAX_VIDEOS]
print(len(videos), "videos,", len(RUN), "strategies:", ", ".join(RUN))

## Choose frames

Each method returns the frame numbers to keep. `scene_change` and `sharpest` first scan the video at low resolution.

In [ ]:
def video_info(path):
    capture = cv2.VideoCapture(str(path))
    fps, total = capture.get(cv2.CAP_PROP_FPS), int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    capture.release()
    if not fps or total <= 0:
        raise ValueError("no readable video stream")
    return fps, total


def scan(path, step, width):
    """Yield (frame number, small grayscale image) for every step-th frame."""
    capture = cv2.VideoCapture(str(path))
    index = 0
    while capture.grab():
        if index % step == 0:
            ok, frame = capture.retrieve()
            if ok:
                height = max(1, round(frame.shape[0] * width / frame.shape[1]))
                yield index, cv2.cvtColor(cv2.resize(frame, (width, height)), cv2.COLOR_BGR2GRAY)
        index += 1
    capture.release()


def scene_changes(path, fps, threshold, ratio, min_gap):
    """Pick frames where the change spikes well above the motion around it."""
    indices, changes, previous = [], [], None
    for index, gray in scan(path, max(1, round(fps / ANALYSIS_FPS)), 160):
        if previous is not None:
            indices.append(index)
            changes.append(float(cv2.absdiff(gray, previous).mean()))
        previous = gray
    picked, changes = [0], np.array(changes)
    for k, (index, change) in enumerate(zip(indices, changes)):
        nearby = np.median(changes[max(0, k - ANALYSIS_FPS):k + ANALYSIS_FPS + 1])
        if change >= threshold and change >= ratio * max(nearby, 1) and (index - picked[-1]) / fps >= min_gap:
            picked.append(index)
    return picked


def sharpest(path, fps, seconds):
    best = {}
    for index, gray in scan(path, max(1, round(fps / ANALYSIS_FPS)), 320):
        window = int(index / fps // seconds)
        score = cv2.Laplacian(gray, cv2.CV_64F).var()
        if window not in best or score > best[window][0]:
            best[window] = (score, index)
    return [index for _, index in best.values()]


def keyframes(path, fps, total):
    result = subprocess.run(
        ["ffprobe", "-v", "error", "-select_streams", "v:0", "-skip_frame", "nokey",
         "-show_entries", "frame=pts_time", "-of", "csv=p=0", str(path)],
        capture_output=True, text=True, timeout=300, check=True,
    )
    times = [float(line.strip(",")) for line in result.stdout.split() if line.strip(",")]
    return [min(round(t * fps), total - 1) for t in times]


def plan(path, config, fps, total):
    method = config["method"]
    if method == "every_n":
        indices = list(range(0, total, config["n"]))
    elif method == "every_seconds":
        steps = int(total / fps / config["seconds"]) + 1
        indices = [round(k * config["seconds"] * fps) for k in range(steps)]
    elif method == "uniform":
        indices = [int((k + 0.5) * total / config["count"]) for k in range(config["count"])]
    elif method == "scene_change":
        indices = scene_changes(path, fps, config["threshold"], config["ratio"], config["min_gap"])
    elif method == "sharpest":
        indices = sharpest(path, fps, config["seconds"])
    elif method == "keyframes":
        indices = keyframes(path, fps, total)
    else:
        raise ValueError(f"unknown method {method}")
    indices = sorted({i for i in indices if 0 <= i < total})
    if len(indices) > MAX_FRAMES_PER_VIDEO:
        keep = np.linspace(0, len(indices) - 1, MAX_FRAMES_PER_VIDEO).round().astype(int)
        indices = [indices[k] for k in keep]
    return indices

## Save frames

Each video gets a folder with its JPEGs and a `frames.json` recording every frame number and timestamp. File names look like `000120_0004.000s.jpg`.

In [ ]:
def difference_hash(frame):
    gray = cv2.cvtColor(cv2.resize(frame, (9, 8)), cv2.COLOR_BGR2GRAY)
    return int("".join("1" if bit else "0" for bit in (gray[:, 1:] > gray[:, :-1]).flatten()), 2)


def extract(path, indices, folder, dedupe):
    wanted, last, rows, last_hash, index = set(indices), max(indices), [], None, 0
    capture = cv2.VideoCapture(str(path))
    while index <= last and capture.grab():
        if index in wanted:
            seconds = capture.get(cv2.CAP_PROP_POS_MSEC) / 1000
            ok, frame = capture.retrieve()
            if ok:
                current = difference_hash(frame)
                if dedupe is None or last_hash is None or bin(current ^ last_hash).count("1") > dedupe:
                    name = f"{index:06d}_{seconds:09.3f}s.jpg"
                    cv2.imwrite(str(folder / name), frame, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
                    rows.append({"file": name, "frame": index, "time": round(seconds, 3)})
                    last_hash = current
        index += 1
    capture.release()
    return rows


def process(video, name, config):
    relative = video.relative_to(RAW)
    folder = FRAMES / name / relative.with_suffix("")
    done = folder / "frames.json"
    if done.exists():
        return "skipped"
    started = time.time()
    if config["method"] == "cover":
        cover = video.with_name(video.stem + "_cover.jpg")
        if not cover.exists():
            raise FileNotFoundError("no cover image")
        folder.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(cover, folder / "cover.jpg")
        fps, total, rows = None, None, [{"file": "cover.jpg", "frame": None, "time": None}]
    else:
        fps, total = video_info(video)
        indices = plan(video, config, fps, total)
        folder.mkdir(parents=True, exist_ok=True)
        rows = extract(video, indices, folder, config.get("dedupe"))
    record = {
        "id": video.stem, "source": str(relative), "strategy": name, "settings": config,
        "fps": fps, "total_frames": total, "seconds_taken": round(time.time() - started, 2), "frames": rows,
    }
    partial = done.with_suffix(".partial")
    partial.write_text(json.dumps(record, indent=1), encoding="utf-8")
    partial.rename(done)
    return "done"

## Run

Failures are collected and don't stop the run.

In [ ]:
errors = []
for name in RUN:
    for video in tqdm(videos, desc=name):
        try:
            process(video, name, CONFIGS[name])
        except Exception as error:
            errors.append({"strategy": name, "source": str(video.relative_to(RAW)), "error": f"{type(error).__name__}: {error}"})

FRAMES.mkdir(parents=True, exist_ok=True)
errors = pd.DataFrame(errors, columns=["strategy", "source", "error"])
errors.to_csv(FRAMES / "errors.csv", index=False)
print(len(errors), "errors")
errors.groupby(["strategy", "error"]).size() if len(errors) else None

## Compare strategies

Writes `summary.csv` comparing the strategies. To work with individual frames, run `04_build_tables` and use `images.csv`.

In [ ]:
summary = []
for name in RUN:
    rows, videos_done, seconds = [], 0, 0.0
    for done in sorted((FRAMES / name).rglob("frames.json")):
        record = json.loads(done.read_text(encoding="utf-8"))
        videos_done += 1
        seconds += record["seconds_taken"]
        for frame in record["frames"]:
            rows.append({
                "id": record["id"], "source": record["source"], "strategy": name,
                "path": str(done.parent / frame["file"]), "frame": frame["frame"], "time": frame["time"],
            })
    manifest = pd.DataFrame(rows, columns=["id", "source", "strategy", "path", "frame", "time"])
    per_video = manifest.groupby("id").size() if len(manifest) else pd.Series(dtype=int)
    size = sum(f.stat().st_size for f in (FRAMES / name).rglob("*.jpg"))
    summary.append({
        "strategy": name, "videos": videos_done, "frames": len(manifest),
        "mean_per_video": round(per_video.mean(), 1) if len(per_video) else 0,
        "max_per_video": int(per_video.max()) if len(per_video) else 0,
        "size_mb": round(size / 1024**2, 1), "seconds": round(seconds, 1),
    })
summary = pd.DataFrame(summary)
summary.to_csv(FRAMES / "summary.csv", index=False)
print(summary.to_string(index=False))
summary